In [17]:
import pingouin

import sys
sys.path.append('../mdi/')
import globals as gl

import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import os
import pandas as pd

plt.style.use('default')

In [18]:
data = pd.read_csv('/home/alily/Documents/GitHub/modified-digit-inteference/mdi/MDI0_merged1.csv')
N = data.SID.nunique()
data = data[(data.correct==1) & (data.BN>1)]
data[['ipi1', 'ipi2', 'ipi3', 'ipi4']] = data[['ipi1', 'ipi2', 'ipi3', 'ipi4']].astype(float)
data.PosInQuartet = pd.Categorical(data.PosInQuartet, categories=[1, 2, 3, 4], ordered=True)
data.Quartet = pd.Categorical(data.Quartet, categories=['AAAA', 'AAMA', 'AARA'], ordered=True)

data.head(6)

,BN,TN,startTR,startTRReal,startTimeReal,planTime,execTime,feedbackTime,iti,expectedDigit1,...,SN,SID,PosInQuartet,Quartet,planError,correct,ipi1,ipi2,ipi3,ipi4
74,2,3,0,15,14471,1000,5000,1000,500,5,...,0,100,3,AAMA,False,True,945.0,700.0,325.0,320.0
75,2,4,0,20,19606,1000,5000,1000,500,5,...,0,100,4,AAMA,False,True,550.0,410.0,365.0,265.0
76,2,5,0,25,24666,1000,5000,1000,500,2,...,0,100,1,AARA,False,True,370.0,425.0,395.0,385.0
79,2,8,0,43,42281,1000,5000,1000,500,2,...,0,100,4,AARA,False,True,825.0,560.0,505.0,555.0
80,2,9,0,48,47901,1000,5000,1000,500,4,...,0,100,1,AARA,False,True,685.0,940.0,570.0,650.0
82,2,11,0,60,59621,1000,5000,1000,500,4,...,0,100,3,AARA,False,True,540.0,640.0,255.0,80.0


In [23]:
condition_data = data[(data.Quartet!="AAAA")]
condition_data.groupby(["SID","Quartet","PosInQuartet"])

anova=pingouin.rm_anova(data=condition_data,dv="MovementTime",subject="SN",within=["Quartet","PosInQuartet"])
display(anova)

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,Quartet,2789.332271,1,16,2789.332271,0.218819,6.462492e-01,6.462492e-01,0.000099,1.000000
1,PosInQuartet,303512.603443,3,48,101170.867814,16.836230,1.314124e-07,1.668043e-07,0.010616,0.981602
2,Quartet * PosInQuartet,10098.745077,3,48,3366.248359,0.538052,6.584925e-01,6.159912e-01,0.000357,0.780828


In [24]:
melted_ipi = condition_data.melt(id_vars=["Quartet", 'SID', 'PosInQuartet', 'TN', 'BN'],value_vars=["ipi1","ipi2","ipi3","ipi4"], value_name="IPI", var_name="IPI_id")
melted_ipi['IPI'] = (melted_ipi['IPI'] - melted_ipi.groupby('SID')['IPI'].transform('mean'))
melted_ipiG = melted_ipi.groupby(['SID', 'Quartet', 'PosInQuartet', 'IPI_id'], observed=True).mean(numeric_only=True).reset_index()
melted_ipiG['IPI'] += melted_ipiG['IPI'].mean()



In [25]:
melted_ipi.head(6)

,Quartet,SID,PosInQuartet,TN,BN,IPI_id,IPI
0,AAMA,100,3,3,2,ipi1,548.738789
1,AAMA,100,4,4,2,ipi1,153.738789
2,AARA,100,1,5,2,ipi1,-26.261211
3,AARA,100,4,8,2,ipi1,428.738789
4,AARA,100,1,9,2,ipi1,288.738789
5,AARA,100,3,11,2,ipi1,143.738789


In [29]:
#ipi_anova = AnovaRM(data=melted_ipi,subject="SID",depvar="IPI",within=["PosInQuartet","IPI_id"],aggregate_func="mean")

ipi_anova=pingouin.rm_anova(data=melted_ipi,dv="IPI",subject="SID",within=["IPI_id","PosInQuartet"])


display(ipi_anova)

/home/alily/Documents/GitHub/modified-digit-inteference/.venv/lib/python3.12/site-packages/pingouin/distribution.py:514: UserWarning: Epsilon values might be innaccurate in two-way repeated measures design where each  factor has more than 2 levels. Please  double-check your results.
  warnings.warn(


,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,IPI_id,126047.056121,3,48,42015.685374,13.330124,1.868277e-06,2.426275e-05,0.391003,0.754964
1,PosInQuartet,28218.568227,3,48,9406.189409,28.338803,1.078804e-10,6.124230e-09,0.125673,0.796964
2,IPI_id * PosInQuartet,3415.019619,9,144,379.446624,1.881313,5.914869e-02,1.687789e-01,0.017098,0.222601
